[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/andyrdt/puzzles/blob/main/09_2026/starter_notebook.ipynb)

# Monthly Algorithmic Challenge — September 2026: Set Difference

*Inspired by Callum McDougall's [ARENA Monthly Algorithmic Challenges](https://learn.arena.education/chapter1_transformer_interp/monthly_algorithmic/).*

## Task

Given a set `X` of four distinct symbols, and the same set with one symbol `z` removed, predict `z`. Both halves are shuffled independently. The released model is correct on every valid prompt. Reverse-engineer the algorithm it learned.

## The model

| | |
|---|---|
| **Symbols** | 16 (rendered as `a` – `p`) |
| **Input format** | `[BOS] x1 x2 x3 x4 [SEP] y1 y2 y3 [SEP]` |
| **Output** | `z`, predicted at the final `SEP` |
| **Layers** | 1 |
| **Heads** | 2 |
| **`d_model`** | 2 |
| **Parameters** | 108 |
| **Vocab** | 18 tokens (16 symbols + `BOS` + `SEP`) |
| **Positional embeddings** | learned |
| **Causal mask** | yes |

Attention-only: no MLPs, no LayerNorm, no biases.

```
token_embedding + positional_embedding
    → attention layer (2 heads, causal mask) + residual
    → linear unembed → logits
```

## Submission

Submit a Colab notebook describing the algorithm.

## Setup

In [ ]:
%pip install -q torch einops nnsight==0.6.3 huggingface_hub matplotlib

In [ ]:
import json, importlib
from pathlib import Path

import torch
import matplotlib.pyplot as plt
from nnsight import NNsight
from huggingface_hub import hf_hub_download

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})

REPO_ID = "andyrdt/09_2026_puzzle_1"
model_py_path = hf_hub_download(REPO_ID, "model.py")
spec = importlib.util.spec_from_file_location("model", model_py_path)
model_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(model_module)
AttentionOnlyTransformer = model_module.AttentionOnlyTransformer

config_path  = hf_hub_download(REPO_ID, "config.json")
weights_path = hf_hub_download(REPO_ID, "model.pt")

config = json.loads(Path(config_path).read_text())
raw_model = AttentionOnlyTransformer.from_config(config["model"])
raw_model.load_state_dict(torch.load(weights_path, map_location=device, weights_only=True))
raw_model.eval().to(device)
model = NNsight(raw_model)

print(f"Model config: {config['model']}")
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

## Vocabulary

| Token id range | Role | Render |
|---|---|---|
| `0` – `15` | Symbols | `a` – `p` |
| `16` | `BOS` | `BOS` |
| `17` | `SEP` | `SEP` |

Input sequence: `[BOS, x1, x2, x3, x4, SEP, y1, y2, y3, SEP]` (length 10). The model predicts `z` at the final `SEP`.

In [ ]:
import random

NUM_SYMBOLS = config["vocab"]["num_symbols"]   # 16
SET_SIZE    = config["task"]["set_size"]       # 4
BOS         = NUM_SYMBOLS                       # 16
SEP         = NUM_SYMBOLS + 1                   # 17

def symbol_to_token(s):
    if isinstance(s, str):
        return ord(s) - ord("a")
    return int(s)

def token_label(tok):
    if 0 <= tok < NUM_SYMBOLS:
        return chr(ord("a") + tok)
    if tok == BOS: return "BOS"
    if tok == SEP: return "SEP"
    return f"?{tok}"

def encode(X, Y):
    assert len(X) == SET_SIZE and len(Y) == SET_SIZE - 1
    return [BOS] + [symbol_to_token(s) for s in X] + [SEP] + [symbol_to_token(s) for s in Y] + [SEP]

def make_prompt(X, z, seed=0):
    # Shuffle X and Y = X \ {z} independently, then encode.
    assert z in X and len(set(X)) == SET_SIZE
    rng = random.Random(seed)
    X = list(X); Y = [s for s in X if s != z]
    rng.shuffle(X); rng.shuffle(Y)
    return encode(X, Y)

tokens = make_prompt(["b", "e", "j", "n"], z="j", seed=0)
print("token ids :", tokens)
print("labels    :", [token_label(t) for t in tokens])
print("answer    : j")

## Running the model with `nnsight`

[`nnsight`](https://nnsight.net/) lets you capture activations and run interventions inside a `with model.trace(x): ...` block:

```python
with model.trace(x):
    out = model.output.save()
logits, attn_patterns = out
```

Intermediate activations work the same way (e.g. `model.layers[0].output.save()`); interventions are in-place mutations of activation tensors inside the block.

## Example outputs

In [ ]:
examples = [
    (["b", "e", "j", "n"], "j"),
    (["a", "c", "k", "p"], "a"),
    (["d", "h", "l", "m"], "m"),
]

fig, axes = plt.subplots(1, len(examples), figsize=(4 * len(examples), 3), sharey=True)
for ax, (X, z) in zip(axes, examples):
    tokens = make_prompt(X, z, seed=0)
    x = torch.tensor([tokens], device=device)

    with model.trace(x):
        out = model.output.save()
    logits, _ = out

    probs = torch.softmax(logits[0, -1, :NUM_SYMBOLS], dim=-1).detach().cpu()
    pred = token_label(int(probs.argmax()))

    ax.bar([token_label(t) for t in range(NUM_SYMBOLS)], probs.numpy(), color="tab:blue")
    ax.set_ylim(0, 1)
    ax.set_yticks([0, 0.5, 1])
    ax.set_title(" ".join(token_label(t) for t in tokens) + f"\npred = {pred}, true = {z}", fontsize=10)
axes[0].set_ylabel("P(symbol)")

plt.suptitle("Output distribution at the final SEP")
plt.tight_layout()
plt.show()

## Token embeddings

With `d_model=2`, every token embedding is a point in the plane.

In [ ]:
E = raw_model.tok_embed.weight.detach().cpu()

fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(E[:NUM_SYMBOLS, 0], E[:NUM_SYMBOLS, 1], s=50, color="tab:blue", label="symbols a–p")
ax.scatter(E[BOS:, 0], E[BOS:, 1], s=50, color="tab:red", marker="s", label="BOS, SEP")
for tok in range(E.shape[0]):
    ax.annotate(token_label(tok), (E[tok, 0], E[tok, 1]), xytext=(6, 0), textcoords="offset points", va="center")
ax.axhline(0, color="gray", lw=0.5, zorder=0); ax.axvline(0, color="gray", lw=0.5, zorder=0)
ax.set_xlabel("dim 0"); ax.set_ylabel("dim 1")
ax.set_title("Token embeddings")
ax.legend(loc="upper left")
plt.show()

## Attention patterns

Attention from the final `SEP` to every position, for one example.

In [ ]:
X, z = ["b", "e", "j", "n"], "j"
tokens = make_prompt(X, z, seed=0)
x = torch.tensor([tokens], device=device)
labels = [token_label(t) for t in tokens]

with model.trace(x):
    out = model.output.save()
_, attn_patterns = out

# attn_patterns[0] has shape (batch, n_heads, query_pos, key_pos) for layer 0.
attn_from_sep = attn_patterns[0][0, :, -1].detach().cpu()   # (n_heads, seq_len)

fig, axes = plt.subplots(1, attn_from_sep.shape[0], figsize=(5 * attn_from_sep.shape[0], 3), sharey=True)
for h, ax in enumerate(axes):
    ax.bar(range(len(labels)), attn_from_sep[h].numpy(), color="tab:blue")
    ax.set_xticks(range(len(labels)), labels)
    ax.set_title(f"Head {h}")
    ax.set_xlabel("Key position")
axes[0].set_ylabel("Attention from final SEP")
plt.suptitle(f"{' '.join(labels)}   (missing symbol: {z})")
plt.tight_layout()
plt.show()

## Weights

Direct access to every weight matrix via `raw_model`.

In [ ]:
for name, p in raw_model.named_parameters():
    print(f"  {name:40s}  shape = {tuple(p.shape)}")

## Your turn

- `model` — `nnsight`-wrapped; use `with model.trace(x): ...` to capture or intervene.
- `raw_model` — plain `nn.Module`; weights accessible directly.